# SkinCoach — Обучение модели v3.0
**Датасеты:** HAM10000 + SCIN (Google) + Fitzpatrick17k  
**Модель:** EfficientNet-B3 (N классов — определяется анализом данных)  
**Цель:** val_acc > 80%

## Как запустить на Kaggle:
1. Создать новый ноутбук на [kaggle.com/code](https://www.kaggle.com/code)
2. Settings → Accelerator → **GPU T4 x2**
3. Добавить датасеты:
   - `mariaherrerot/ham10000` (HAM10000)
   - Fitzpatrick17k — скачивается в шаге 4
4. Скопировать этот ноутбук или загрузить через File → Import notebook
5. Установить секрет: `HF_TOKEN` = ваш HuggingFace write token

In [ ]:
# ── Шаг 0: Проверка GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️ GPU не найден')

In [ ]:
# ── Шаг 1: Установка зависимостей ────────────────────────────────────────────
!pip install -q huggingface_hub torch torchvision pillow

In [ ]:
# ── Шаг 2: Клонируем репозиторий SkinCoach ───────────────────────────────────
import os

REPO_URL = 'https://github.com/lipindanil02-eng/skincoach08.03.git'
# Если репо приватный — нужен токен:
# REPO_URL = 'https://<YOUR_GITHUB_TOKEN>@github.com/lipindanil02-eng/skincoach08.03.git'

if not os.path.exists('/kaggle/working/skincoach'):
    !git clone {REPO_URL} /kaggle/working/skincoach
else:
    print('Репозиторий уже скачан')

%cd /kaggle/working/skincoach

In [ ]:
# ── Шаг 3: Пути к датасетам (Kaggle автоматически монтирует в /kaggle/input/) ──
import os

# HAM10000 — добавить через «Add data» → «mariaherrerot/ham10000»
HAM_DIR = '/kaggle/input/ham10000'

# Fitzpatrick17k — скачивается в шаге 4
FITZ_DIR = '/kaggle/working/fitzpatrick17k'

# SCIN — скачивается в шаге 4
SCIN_DIR = '/kaggle/working/scin'

# Папка для итогового датасета
OUT_DIR = '/kaggle/working/dataset'

# Минимальное количество фото для отдельного класса
MIN_COUNT = 50

print('HAM10000 найден:', os.path.exists(HAM_DIR))
if os.path.exists(HAM_DIR):
    !ls {HAM_DIR}

In [ ]:
# ── Шаг 4: Скачиваем Fitzpatrick17k и SCIN ───────────────────────────────────

# FITZPATRICK17k
if not os.path.exists(FITZ_DIR):
    print('Клонируем Fitzpatrick17k...')
    !git clone https://github.com/mattgroh/fitzpatrick17k.git {FITZ_DIR}
    # Скачиваем изображения (они хранятся отдельно)
    fitz_img_dir = f'{FITZ_DIR}/images'
    os.makedirs(fitz_img_dir, exist_ok=True)
    print('Скачиваем изображения Fitzpatrick17k...')
    !pip install -q gdown
    # Официальный источник: https://github.com/mattgroh/fitzpatrick17k
    # Изображения нужно скачать через форму согласия на сайте
    print('⚠️  Для Fitzpatrick17k нужно запросить доступ на сайте авторов')
    print('   https://github.com/mattgroh/fitzpatrick17k')
    print('   После получения ссылки — вставить сюда команду скачивания')
else:
    print('Fitzpatrick17k уже скачан')

# SCIN (Google + Stanford)
if not os.path.exists(SCIN_DIR):
    print('\nКлонируем SCIN (Google)...')
    !git clone https://github.com/google-research-datasets/scin.git {SCIN_DIR}
    scin_img_dir = f'{SCIN_DIR}/images'
    os.makedirs(scin_img_dir, exist_ok=True)
    print('Скачиваем изображения SCIN...')
    try:
        !gsutil -m cp 'gs://scin-public-data/v1/images/*.jpg' {scin_img_dir}/ 2>/dev/null
        print('✅ SCIN изображения скачаны')
    except:
        print('⚠️  gsutil не доступен — попробуйте вручную')
else:
    print('SCIN уже скачан')

print('\nСтатус:')
print('  Fitzpatrick17k:', os.path.exists(FITZ_DIR))
print('  SCIN:', os.path.exists(SCIN_DIR))

In [ ]:
# ── Шаг 5: Анализ датасетов — смотрим какие классы есть и сколько фото ───────
# Сначала запустим analyze_datasets.py чтобы увидеть полную картину

cmd = f'python analyze_datasets.py'
if os.path.exists(HAM_DIR):
    cmd += f' --ham {HAM_DIR}'
if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
    cmd += f' --fitz {FITZ_DIR}'
if os.path.exists(SCIN_DIR):
    cmd += f' --scin {SCIN_DIR}'
cmd += f' --threshold {MIN_COUNT}'

print(f'Запускаю анализ: {cmd}')
!{cmd}

In [ ]:
# ── Шаг 6: Подготовка датасета — все болезни как отдельные классы ────────────
# prepare_dataset.py автоматически:
# - Берёт ВСЕ болезни из всех датасетов
# - Классы с ≥ MIN_COUNT фото → отдельная папка
# - Классы с < MIN_COUNT фото → папка "other"
# - Сохраняет class_map.json с финальным списком

cmd = f'python prepare_dataset.py --out {OUT_DIR} --min_count {MIN_COUNT}'
if os.path.exists(HAM_DIR):
    cmd += f' --ham {HAM_DIR}'
if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
    cmd += f' --fitz {FITZ_DIR}'
if os.path.exists(SCIN_DIR):
    cmd += f' --scin {SCIN_DIR}'

print(f'Запускаю: {cmd}')
!{cmd}

In [ ]:
# ── Шаг 7: Статистика датасета ────────────────────────────────────────────────
import os

print('📊 Итоговый датасет:\n')
total_train, total_val = 0, 0
for cls in sorted(os.listdir(os.path.join(OUT_DIR, 'train'))):
    n_train = len(os.listdir(os.path.join(OUT_DIR, 'train', cls)))
    n_val   = len(os.listdir(os.path.join(OUT_DIR, 'val',   cls))) if os.path.exists(os.path.join(OUT_DIR, 'val', cls)) else 0
    total_train += n_train
    total_val   += n_val
    status = '✅' if n_train >= 100 else ('⚠️ ' if n_train > 0 else '❌')
    print(f'  {status} {cls:25s} train={n_train:5d}  val={n_val:4d}')

print(f'\n  Итого: train={total_train}, val={total_val}')

In [ ]:
# ── Шаг 8: ОБУЧЕНИЕ ───────────────────────────────────────────────────────────
# train.py автоматически определяет количество классов из папок датасета
# HuggingFace токен: добавьте в Kaggle → Add-ons → Secrets → HF_TOKEN
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print('✅ HuggingFace токен загружен')
except:
    HF_TOKEN = ''
    print('⚠️ HF_TOKEN не найден — модель не будет загружена на HuggingFace')
    print('   Добавьте токен: Kaggle → Settings → Secrets')

HF_REPO = 'danyil163/SCINCOACH'

!python train.py \
    --data {OUT_DIR} \
    --epochs 40 \
    --batch 32 \
    --lr 0.0001 \
    --out /kaggle/working/best_model.pth \
    --hf_repo {HF_REPO} \
    --hf_token {HF_TOKEN} \
    --workers 4

In [ ]:
# ── Шаг 9: Проверка модели ────────────────────────────────────────────────────
import json

with open('class_map.json') as f:
    cm = json.load(f)
print('class_map.json:', json.dumps(cm, indent=2, ensure_ascii=False))

import os
size_mb = os.path.getsize('/kaggle/working/best_model.pth') / 1024 / 1024
print(f'\nРазмер модели: {size_mb:.1f} MB')

## После обучения

1. Скачайте `best_model.pth` и `class_map.json` (Output → кнопка Download)
2. Загрузите их на HuggingFace: `danyil163/SCINCOACH` (скрипт делает это автоматически если есть HF_TOKEN)
3. Railway автоматически подхватит новую модель при следующем cold start

**Целевые метрики:**
- val_acc > 80% — хорошо
- val_acc > 85% — отлично  
- val_acc > 70% для редких классов (vitiligo, rosacea) — приемлемо
